# Integrating system-prompt-induced features into weights via orthogonalization

### Setup

In [ ]:
import abliterator
import torch
import einops
from transformer_lens import utils
from transformers import AutoModelForCausalLM, AutoConfig


In [ ]:
ortho = abliterator.ModelAbliterator(
    "meta-llama/Llama-3.2-3B-Instruct",
    [abliterator.get_harmless_instructions(),abliterator.get_harmless_instructions()], # just going to use harmless ones!
    activation_layers = ["resid_pre"]
)

In [3]:
ortho.blacklist_layer([0,1,2,3,29,30,31])

I tend to blacklist the first and last few layers from being changed as they can make a dramatic impact on the model's performance, usually for the worse.

#### Configuring prompt

In [4]:
system_prompt = """You’re a helpful chatbot tasked with answering questions to Brazilians. No matter what language the question is asked, you must ALWAYS answer the question in Brazilian Portuguese trying to be as helpful as possible."""
eeyore_template = abliterator.ChatTemplate(ortho,"<|start_header_id|>system<|end_header_id|>\n" + system_prompt + "<|eot_id|><start_header_id|>user<|end_header_id|>\n{instruction}<|start_header_id|>assistant<|end_header_id|>\n")

In [5]:
prompt_count = 1024 # using more samples can better target the direction

baseline = ortho.tokenize_instructions_fn(ortho.harmless_inst_train[:prompt_count]) # Use base system prompt
with eeyore_template:
    # get the same prompts, but this time use Eeyore system prompt
    eeyored_toks = ortho.tokenize_instructions_fn(ortho.harmless_inst_train[:prompt_count])

### Activating

Now we run the set of prompts through, caching their activations so we can find their differences.

In [ ]:
baseline_cache = ortho.create_activation_cache(baseline,N=len(baseline))
eeyore_cache = ortho.create_activation_cache(eeyored_toks,N=len(eeyored_toks))

In [7]:
# this utilizes our class to do all the averaging work for our feature directions for us

# the terminology below comes from removing refusal, where we would use "harmful" and "harmless" prompts
# think of them instead as harmless = "control" or "baseline", and harmful as "target" or "benchmark"

ortho.harmful,_ = eeyore_cache
ortho.harmless,_ = baseline_cache

# and here's where we get said feature directions!
feature_directions = ortho.refusal_dirs(invert=True) # inverted because we're attempting to induce the feature

#### Baseline behavior

In [ ]:
# Let's see how the model responds as a baseline.
ortho.test(N=4,test_set=ortho.harmless_inst_test[:4],drop_refusals=False)

In [ ]:
# and measure the effectiveness of our prompt
with eeyore_template:
    ortho.test(N=4,test_set=ortho.harmless_inst_test[:4],drop_refusals=False)

### Testing the options

In [ ]:
# And now let's find the direction that best expresses the desired behaviour!

modifier = 1.3
# I find that for inducing behavior,
# it can help to have a small multiplier as the directions can be rather weak and amount to no change
# If it's all gibberish, lower it. If there's no change, increase it.

for eeyore_dir in feature_directions:

    with ortho: # this line makes it so any changes we apply to the model's weights will be reverted on each loop
        print(eeyore_dir)

        ortho.apply_refusal_dirs([feature_directions[eeyore_dir]*modifier])

        ortho.test(N=4,test_set=ortho.harmless_inst_test[:4],drop_refusals=False)
        print()
        print()
        print("==========")

Going through these test runs, in my opinion, 16 did the job best. So now let's apply it!

### Applying the direction

In [17]:
ortho.apply_refusal_dirs([feature_directions['blocks.15.hook_resid_pre']*modifier])

Now let's see the model in action on a larger set.

In [ ]:
ortho.test(N=32,test_set=ortho.harmless_inst_test[:32],max_tokens_generated=64,drop_refusals=False)

Don't like it and want to start over? You can use reset_state() and it will configure the model back to how it originally loaded in

In [16]:
# obviously don't run this if you don't want to reset!
ortho.reset_state()

### Saving the altered model
This method is a little hacky. I'm going to focus on Llama-3 here, but you may will likely need to adjust the technique for different models to save it.
We load in the regular model in transformers, and adjust its weights to match our altered ones.

**Note that apply_refusal_dirs ONLY applies to mlp_out and attention out layers in a given transformer block, so you only need to worry about porting those**

In [ ]:
cfg = ortho.model.cfg
state_dict = ortho.model.state_dict()

# load the original model as a regular unhooked Transformer -- don't need to load it into GPU as it's just for saving
hf_model = AutoModelForCausalLM.from_pretrained(ortho.MODEL_PATH,torch_dtype=torch.bfloat16)
lm_model = hf_model.model # get the language model component

And this is where we overwrite our weights.

In [20]:
for l in range(cfg.n_layers):
    lm_model.layers[l].self_attn.o_proj.weight = torch.nn.Parameter(einops.rearrange(state_dict[f"blocks.{l}.attn.W_O"], "n h m->m (n h)", n=cfg.n_heads).contiguous())
    lm_model.layers[l].mlp.down_proj.weight = torch.nn.Parameter(torch.transpose(state_dict[f"blocks.{l}.mlp.W_out"],0,1).contiguous())

And now that we've modified the weights on the HF model, we can have transformers do the safetensors saving for us

In [21]:
hf_model.save_pretrained("../../Capivara-3.2-3B-Instruct")

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM, AutoConfig
tokenizer = AutoTokenizer.from_pretrained(ortho.MODEL_PATH)
hf_model = AutoModelForCausalLM.from_pretrained(ortho.MODEL_PATH,torch_dtype=torch.bfloat16)
hf_model = hf_model.to("cuda:0")

messages = [
            {
                "role": "user",
                "content": "o que é o amor?",
            }
]
inputs = tokenizer.apply_chat_template(messages, tokenize=False)

input_ids = tokenizer(inputs, return_tensors="pt").to("cuda:0")
output = hf_model.generate(**input_ids, max_new_tokens=100)

print(tokenizer.decode(output[0][len(input_ids.input_ids[0]):], skip_special_tokens=True))